# 03 — Evaluate Calendar vs. Weather Models

**Pipeline stage:** does weather actually add predictive value, beyond what calendar patterns alone
already explain?

This is the paper's central methodological move (Research Question 1). High accuracy alone doesn't
prove weather matters — hour-of-day, day-of-week, and month already explain a lot of a country's
renewable-share pattern. So every comparison here is **calendar-only vs. calendar-plus-weather**,
same model, same split, same everything else, and the *gain* is what's reported.

`weather_informed/evaluate.py` implements this protocol once, so every downstream script (bootstrap,
figures, resolution sweep) reuses exactly the same feature sets, models, and split logic instead of
each reimplementing it slightly differently.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from weather_informed.evaluate import CALENDAR_COLUMNS, WEATHER_COLUMNS, SHARE_COL, make_features

print("Target column:     ", SHARE_COL)
print("Calendar features: ", CALENDAR_COLUMNS)
print("Weather features:  ", WEATHER_COLUMNS)

Target column:      Renewable_share_of_load
Calendar features:  ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'is_weekend']
Weather features:   ['wind_speed_100m', 'shortwave_radiation', 'temperature_2m']


## Calendar features, for real, on a tiny synthetic frame

`make_features` is the *actual* production function — not a reimplementation — run here on a small
made-up frame just so its output is visible without needing real staged data.

In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
hours = pd.date_range("2024-01-01", periods=72, freq="h", tz="UTC")
toy = pd.DataFrame(
    {
        "wind_speed_100m": rng.uniform(2, 12, len(hours)),
        "shortwave_radiation": rng.uniform(0, 400, len(hours)),
        "temperature_2m": rng.uniform(-5, 15, len(hours)),
        SHARE_COL: rng.uniform(10, 90, len(hours)),
    },
    index=hours,
)

featured = make_features(toy)
featured[["hour_sin", "hour_cos", "month_sin", "month_cos", "is_weekend"]].head()

,hour_sin,hour_cos,month_sin,month_cos,is_weekend
2024-01-01 00:00:00+00:00,0.000000,1.000000,0.5,0.866025,0
2024-01-01 01:00:00+00:00,0.258819,0.965926,0.5,0.866025,0
2024-01-01 02:00:00+00:00,0.500000,0.866025,0.5,0.866025,0
2024-01-01 03:00:00+00:00,0.707107,0.707107,0.5,0.866025,0
2024-01-01 04:00:00+00:00,0.866025,0.500000,0.5,0.866025,0


## Reproducing the calendar-vs-weather comparison mechanics

Below is a **synthetic demonstration**, not a real per-country result — it uses made-up data so it can
run without staged ERA5/Energy-Charts files, but it exercises the exact model classes, hyperparameters,
and chronological 80/20 split that `weather_informed/evaluate.py` uses for the real study
(`RandomForestClassifier(n_estimators=300, random_state=42)` for classification,
`HistGradientBoostingRegressor(random_state=42)` for regression, no shuffling).

To make the demonstration meaningful, the synthetic target is constructed so that weather *should*
help: the classification/regression targets are partly driven by the synthetic wind-speed column, with
calendar patterns and noise layered on top. **Real per-country gains from the actual study are reported
in the paper (median AUC 0.86–0.95, median R² 0.56–0.67 at 0.25°, capacity-weighted) — the numbers this
cell prints are illustrative only.**

In [3]:
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, roc_auc_score

rng = np.random.default_rng(20260723)
n = 24 * 400  # ~400 days of synthetic hourly data
hours = pd.date_range("2024-01-01", periods=n, freq="h", tz="UTC")
wind = rng.uniform(2, 14, n)
share = (
    30
    + 2.5 * wind                                     # weather signal
    + 10 * np.sin(2 * np.pi * hours.hour / 24)        # calendar signal
    + rng.normal(0, 8, n)                             # noise
)
share = np.clip(share, 0, 100)

synthetic = pd.DataFrame(
    {
        "wind_speed_100m": wind,
        "shortwave_radiation": rng.uniform(0, 400, n),
        "temperature_2m": rng.uniform(-5, 20, n),
        SHARE_COL: share,
    },
    index=hours,
)
data = make_features(synthetic)
split = int(len(data) * 0.8)
train, test = data.iloc[:split], data.iloc[split:]

y_train_class = (train[SHARE_COL] > 50).astype(int).to_numpy()
y_test_class = (test[SHARE_COL] > 50).astype(int).to_numpy()

def auc_for(columns):
    model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=1)
    model.fit(train[columns].to_numpy(), y_train_class)
    return roc_auc_score(y_test_class, model.predict_proba(test[columns].to_numpy())[:, 1])

def r2_for(columns):
    model = HistGradientBoostingRegressor(random_state=42)
    model.fit(train[columns], train[SHARE_COL])
    return r2_score(test[SHARE_COL], model.predict(test[columns]))

auc_calendar = auc_for(CALENDAR_COLUMNS)
auc_both = auc_for(WEATHER_COLUMNS + CALENDAR_COLUMNS)
r2_calendar = r2_for(CALENDAR_COLUMNS)
r2_both = r2_for(WEATHER_COLUMNS + CALENDAR_COLUMNS)

print(f"[synthetic demo] Random Forest AUC:   calendar-only={auc_calendar:.3f}  "
      f"calendar+weather={auc_both:.3f}  gain={auc_both - auc_calendar:+.3f}")
print(f"[synthetic demo] Gradient Boosting R2: calendar-only={r2_calendar:.3f}  "
      f"calendar+weather={r2_both:.3f}  gain={r2_both - r2_calendar:+.3f}")
print()
print("Positive gains here confirm the mechanics work end-to-end; they are NOT the paper's results.")

[synthetic demo] Random Forest AUC:   calendar-only=0.691  calendar+weather=0.891  gain=+0.200
[synthetic demo] Gradient Boosting R2: calendar-only=0.243  calendar+weather=0.651  gain=+0.408

Positive gains here confirm the mechanics work end-to-end; they are NOT the paper's results.


In [4]:
!python ../weather_informed/evaluate.py --help

usage: evaluate.py [-h] --code CODE --weather-csv WEATHER_CSV [--split SPLIT]
                   [--data-dir DATA_DIR] [--output OUTPUT]

options:
  -h, --help            show this help message and exit
  --code CODE
  --weather-csv WEATHER_CSV
  --split SPLIT
  --data-dir DATA_DIR
  --output OUTPUT


**DATA CELL — not run here.** A real single-country evaluation looks like:

```bash
python weather_informed/evaluate.py --code de \
    --weather-csv data/country_weather_post_covid/weather_era5_de_capacity_1.0deg.csv \
    --output results/post_covid_spatial_resolution/model_results_de.csv
```

which requires the country's merged energy/target file from notebook 01 and its weather CSV from
notebook 02 to already exist under `data/`.

**Next:** [04 — Resolution and Capacity Sensitivity](04_resolution_and_capacity_sensitivity.ipynb)
looks at how this gain changes across countries, resolutions, and weighting schemes, rather than for
one country at a time.